# ZuCo text + EEG sentiment experiments

This notebook is only the Colab runner. The experiment logic lives in the Python files in the repository. It mounts Drive, prepares paths, builds the reusable classical-feature cache, validates alignment, runs a short smoke test, launches the versioned experiment suite, and displays the results saved to Drive.

The full default suite compares frozen and fine-tuned LaBSE text baselines, EEG only, concatenation, gated fusion, shuffled EEG, and random-noise EEG over three seeds and five sentence-level folds. Finished setup/seed files are skipped when a run is resumed.

## 1. Runtime and Drive

This section checks whether Colab has a GPU and mounts Drive. The original ZuCo files, the reusable feature cache, and every experiment result remain on Drive.

In [ ]:
import os
import torch
from google.colab import drive

drive.mount('/content/drive')
print('torch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'not available')

## 2. Repository and dependencies

This section clones the repository on the first run and pulls the latest commit on later runs. It then installs the Python requirements. No experiment outputs are written into the repository.

In [ ]:
REPO_URL = 'https://github.com/parmisbathaeiyan/zuco-multimodal-sentiment.git'
REPO_DIR = '/content/zuco-multimodal-sentiment'

if not os.path.exists(REPO_DIR):
    !git clone $REPO_URL $REPO_DIR
else:
    !git -C $REPO_DIR pull --ff-only
%cd $REPO_DIR
!pip install -q -r requirements.txt

## 3. Paths and run version

This section defines the Drive data and output locations. The raw ZuCo folder contains both the subject `.mat` files and the fixed labels CSV. `FEATURE_TAG` versions the extracted EEG definition. `RUN_TAG` versions one experiment configuration. Re-running the same `RUN_TAG` resumes missing work; change it when the architecture or training settings change.

In [ ]:
DATA_ROOT = '/content/drive/MyDrive/Thesis/Data'
MAT_DIR = f'{DATA_ROOT}/zuco_og_raw'
LABELS_CSV = f'{MAT_DIR}/zuco_sentiment_labels_task1_fixed.csv'

FEATURES_BASE = f'{DATA_ROOT}/zuco_classical_features'
FEATURE_TAG = 'classical_v2_normalized_line_length'
FEATURES_DIR = f'{FEATURES_BASE}/{FEATURE_TAG}'

RESULTS_BASE = '/content/drive/MyDrive/Thesis/Results/zuco_multimodal_sentiment'
RUN_TAG = 'v1_full'  # change this for a genuinely new experiment version
RUN_DIR = f'{RESULTS_BASE}/{RUN_TAG}'

SETUPS = [
    'text_frozen',
    'text_finetune',
    'eeg_only',
    'concat_finetune',
    'gated_finetune',
    'gated_shuffled_finetune',
    'gated_noise_finetune',
]
SEEDS = [42, 52, 62]

os.makedirs(FEATURES_DIR, exist_ok=True)
os.makedirs(RESULTS_BASE, exist_ok=True)
assert os.path.exists(LABELS_CSV), f'Missing labels file: {LABELS_CSV}'
print('labels:', LABELS_CSV)
print('feature cache:', FEATURES_DIR)
print('run results:', RUN_DIR)

## 4. Build or resume the EEG feature cache

This section reads each large `results*_SR.mat` file and saves one much smaller subject `.npz` file to the versioned Drive cache. It is the slow data-preparation step, but completed subjects are skipped on later runs. The default uses duration-normalized line length.

In [ ]:
!python extract_features.py \
    --mat-dir "$MAT_DIR" \
    --labels-csv "$LABELS_CSV" \
    --out-dir "$FEATURES_DIR" \
    --line-length normalized

## 5. Validate text/EEG alignment

This section checks feature widths, sentence IDs, labels, subject counts, and missing recordings before training. It should report 400 unique sentences and the available subject-sentence rows.

In [ ]:
!python inspect_data.py --labels-csv "$LABELS_CSV" --features-dir "$FEATURES_DIR"

## 6. Short smoke test

This optional run trains the EEG branch for one epoch over two folds. It catches data-shape and saving errors quickly before the expensive LaBSE suite. Its output uses a separate run tag and does not mix with the full results.

In [ ]:
SMOKE_TAG = f'{RUN_TAG}_smoke'
!python run.py \
    --labels-csv "$LABELS_CSV" \
    --features-dir "$FEATURES_DIR" \
    --results-base "$RESULTS_BASE" \
    --run-tag "$SMOKE_TAG" \
    --setups eeg_only \
    --seeds 42 \
    --n-folds 2 \
    --epochs 1 \
    --patience 1 \
    --bootstrap-samples 200

## 7. Full versioned experiment suite

This section runs every setup in `SETUPS` over every seed in `SEEDS` using five sentence-level folds. Each setup/seed JSON is saved as soon as it finishes. The full suite is substantial, but it can be interrupted and resumed by rerunning this cell with the same `RUN_TAG`.

In [ ]:
SETUP_ARGS = ' '.join(SETUPS)
SEED_ARGS = ' '.join(str(seed) for seed in SEEDS)
!python run.py \
    --labels-csv "$LABELS_CSV" \
    --features-dir "$FEATURES_DIR" \
    --results-base "$RESULTS_BASE" \
    --run-tag "$RUN_TAG" \
    --setups $SETUP_ARGS \
    --seeds $SEED_ARGS \
    --n-folds 5 \
    --epochs 12 \
    --patience 4

## 8. Saved summary and plots

This section reads the final artifacts from Drive. The table reports mean out-of-fold accuracy and macro-F1 across seeds, plus paired bootstrap deltas for fusion models. The figures compare scores and confusion matrices.

In [ ]:
from IPython.display import Image, Markdown, display

summary_path = os.path.join(RUN_DIR, 'tables', 'summary.md')
if os.path.exists(summary_path):
    display(Markdown(open(summary_path).read()))
else:
    print('No summary yet:', summary_path)

for name in ['scores.png', 'confusions.png']:
    path = os.path.join(RUN_DIR, 'plots', name)
    if os.path.exists(path):
        display(Image(path))

## 9. Rebuild reports without retraining

Use this cell after changing only plotting or reporting code. It regenerates the summary tables and figures from the saved per-seed JSON files and does not load or train a model.

In [ ]:
!python plot_results.py --run-dir "$RUN_DIR"